In [1]:
from FIT_python.plot_style import apply_style
apply_style()


In [1]:
# %% 
import time
from pathlib import Path
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
from tqdm_joblib import tqdm_joblib
from sklearn.model_selection import ParameterSampler
from FIT_python.config import SPLITS_DIR ,RESULTS_DATA_DIR
from sklearn.model_selection import ParameterSampler
from FIT_python.data_split_and_summary.data_import_utils import load_and_prep_df_individual, get_feature_cols
from FIT_python.pipeline_individual_id.pairwise_individual_id_pipeline import (
    run_all_pairwise_projections_parallel
)
from FIT_python.pipeline_individual_id.generate_trails_and_trailpairs import (
    generate_pairwise_comparisons_from_df
)

import matplotlib.pyplot as plt
import seaborn as sns

# %% 1) Basis-Pfade & Hyperparameter-Räume
splits_root = Path("data/splits/eurasian_otter")
models_root = RESULTS_DATA_DIR /  "eurasian_otter_random_search_standard_metrics" / "best_maj_test_pct"
output_root = Path("results/data/pairwise_comp_eurasian_otter")
output_root.mkdir(parents=True, exist_ok=True)

feature_counts    = [2,5,10,15,50,20,50,100,200]
selection_methods = ["random_forest","forward","univariate","variance","lasso"]
reducers          = ["pca","lda","umap"]
n_components_list = [2, 3, 5, 10]
outlier_methods   = [None,"clip","zscore"]
scaler_methods    = [None,"standard","robust"]
use_sex_opts      = [False, True]
n_jobs            = -1

# 3) Zeig Dir zur Sicherheit an, ob die Datei wirklich da ist:
model_fp = models_root / "eurasian_otter.joblib"
print(model_fp)           # Pfad
print(model_fp.exists())  # True, wenn vorhanden


# 1) Zufälliges Sampling aus dem Parameter-Space
param_dist = {
    "feature_count":    feature_counts,
    "selection_method": selection_methods,
    "reducer":          reducers,
    "n_components":     n_components_list,
    "outlier_method":   outlier_methods,
    "scaler_method":    scaler_methods,
    "use_sex":          use_sex_opts
}

# 100 zufällige Parameter-Kombinationen
sampled_grid = (
    pd.DataFrame(
        list(ParameterSampler(param_dist, n_iter=5, random_state=42))
    )
    .sort_values("feature_count", ascending=False)
    .reset_index(drop=True)
)

# 2) Zusätzliche Standard-Methoden festlegen
std_feature_counts   = [10, 12, 14, 16, 18, 20]
std_selection_method = "forward"
std_reducer          = "lda"
std_n_components     = 2     # bei LDA nicht relevant
std_outlier_method   = None
std_scaler_method    = None

std_df = (
    pd.DataFrame({"feature_count": std_feature_counts})
      .assign(
          selection_method=std_selection_method,
          reducer=std_reducer,
          n_components=std_n_components,
          outlier_method=std_outlier_method,
          scaler_method=std_scaler_method,
      )
      .merge(pd.DataFrame({"use_sex": [False, True]}), how="cross")
)

# 3) Standard-Methoden anhängen & sortieren
sampled_grid = pd.concat([sampled_grid, std_df], ignore_index=True)
sampled_grid.sort_values("feature_count", ascending=False, inplace=True)
sampled_grid.reset_index(drop=True, inplace=True)

# Überprüfen
print(f"Sampled grid shape: {sampled_grid.shape}")
sampled_grid.head()



# 6) Loop über Spezies
species_dir = Path(SPLITS_DIR)/"eurasian_otter"
species = "eurasian_otter"
print(f"\n🔄 Processing species: {species}")

# 6.1) Train/Test laden & Fold entfernen
df = load_and_prep_df_individual(species, Path(SPLITS_DIR)) #falls hier bugs valid vales f und m checken


# 3.2) Feature-Spalten bestimmen
feature_cols = get_feature_cols(df)
print(f"  → {len(feature_cols)} numerische Features")

# direkt vor generate_pairwise_comparisons_from_df
print("Unique individual_id:", df["individual_id"].nunique())
print("Unique trail (fallback_col):", df["trail"].nunique())

# 3.3) Pairs & Summary generieren
comps, summary = generate_pairwise_comparisons_from_df(df,num_individuals=5
)


print("  Summary der Trail-Längen:")
display(summary)

# 3.4) Resume-Logik für species-spezifisches Parquet
results_fp = output_root / f"{species}_all_results_combined.parquet"
if results_fp.exists():
    done = pd.read_parquet(results_fp, columns=sampled_grid.columns)
    done_keys = set(tuple(r) for r in done.values)
    todo = sampled_grid[
        sampled_grid.apply(lambda row: tuple(row.values) not in done_keys, axis=1)
    ].reset_index(drop=True)
else:
    todo = sampled_grid.copy()

print(f"  → Noch offen: {len(todo)}/{len(sampled_grid)} Runs")

# 3.5) Hyper-Parameter-Loop
for idx, row in todo.iterrows():
    # Spalten heißen jetzt "feature_count" (nicht feature_counts)
    n_features      = int(row.feature_count)
    selection       = row.selection_method
    reducer         = row.reducer
    n_components    = int(row.n_components)
    outlier_method  = row.outlier_method
    scaler_method   = row.scaler_method
    use_sex_flag    = bool(row.use_sex)

    desc = (
        f"FC={n_features}|FS={selection}|Red={reducer}/{n_components}"
        f"|Out={outlier_method}|Scl={scaler_method}|Sex={'on' if use_sex_flag else 'off'}"
    )
    print(f"    → Run {idx+1}/{len(todo)}: {desc}")
    t0 = time.time()

    # Outlier- und Scaler-Parameter in Listen verpacken
    outlier_list = [outlier_method] if outlier_method is not None else [None]
    scaler_list  = [scaler_method]  if scaler_method  is not None else [None]

    with tqdm_joblib(tqdm(desc=desc, total=len(comps), leave=False)):
        results = run_all_pairwise_projections_parallel(
            comparisons=comps,
            df=df,
            feature_cols=feature_cols,
            k_features=n_features,
            selection_method=selection,
            reducers=[reducer],
            n_components=[n_components],
            outlier_methods=outlier_list,
            scaler_methods=scaler_list,
            use_sexmodel_prediction=use_sex_flag,
            sexmodel_path=(models_root / f"{species}.joblib") if use_sex_flag else None,
            debug=False,
            n_jobs=n_jobs
        )
    print(f"      → {len(results)} Rows in {time.time()-t0:.1f}s")

    # 3.6) Results anreichern & speichern
    df_run = pd.DataFrame(results)
    for col in sampled_grid.columns:
        df_run[col] = row[col]
    if results_fp.exists():
        df_full = pd.concat([pd.read_parquet(results_fp), df_run], ignore_index=True)
    else:
        df_full = df_run
    df_full.to_parquet(results_fp, index=False)

print(f"  ✅ Fertig für {species}, Parquet: {results_fp}")

# %% 4) Ergebnisse
all_parquets = [results_fp]
df_all = pd.concat((pd.read_parquet(p) for p in all_parquets), ignore_index=True)
df_all.to_parquet(output_root / 'eurasian_otter_all_results_combined.parquet', index=False)
print("\n🎉 Fertig – Ergebnis-Parquet erzeugt:")
print(f"   {output_root/'eurasian_otter_all_results_combined.parquet'}")
display(df_all.head())


c:\Users\Frede\anaconda3\envs\FIT_python_conda_env\Lib\site-packages\tqdm_joblib\__init__.py:4: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


C:\Users\Frede\OneDrive\Projects and Disschaptors\FIT_package\results\data\eurasian_otter_random_search_standard_metrics\best_maj_test_pct\eurasian_otter.joblib
True
Sampled grid shape: (17, 7)

🔄 Processing species: eurasian_otter
  → 205 numerische Features
Unique individual_id: 27
Unique trail (fallback_col): 54
  Summary der Trail-Längen:


,sub_size,n_animals,n_trails,avg_comp_per_ind_same,sd_comp_per_ind_same,avg_comp_per_ind_diff,sd_comp_per_ind_diff,ratio_same_to_diff
0,10,5,25,NaN,NaN,NaN,NaN,NaN
1,Total,5,300,10.0,0.0,100.0,0.0,50.0


  → Noch offen: 17/17 Runs
    → Run 1/17: FC=50|FS=random_forest|Red=umap/5|Out=zscore|Scl=standard|Sex=off


FC=50|FS=random_forest|Red=umap/5|Out=zscore|Scl=standard|Sex=off:   0%|          | 0/300 [00:00<?, ?it/s]

  0%|          | 0/300 [00:00<?, ?it/s]

  0%|          | 0/300 [00:00<?, ?it/s]

      → 300 Rows in 43.5s


Processing Pairs:   0%|          | 0/300 [00:43<?, ?it/s]


ArrowTypeError: ("Expected bytes, got a 'bool' object", 'Conversion failed for column same_individual with type object')

In [4]:
df_all

,trail_a_id,trail_b_id,samples_a,samples_b,ind_a,ind_b,same_individual,fold,pipeline,selection_method,...,mean_mahalanobis_within_b,median_mahalanobis_within_b,use_sex,feature_count,avg_proba_A_0,avg_proba_A_1,avg_proba_B_0,avg_proba_B_1,avg_proba_R_0,avg_proba_R_1
0,shorfy_1c0,rhianna_1c0,[228],[308],shorfy,rhianna,False,2,random_forest_zscore_standard_k50_umap_nc5_sex...,random_forest,...,None,None,False,50,NaN,NaN,NaN,NaN,NaN,NaN
1,shorfy_1c0,rhianna_1c0,[228],[317],shorfy,rhianna,False,2,random_forest_zscore_standard_k50_umap_nc5_sex...,random_forest,...,None,None,False,50,NaN,NaN,NaN,NaN,NaN,NaN
2,shorfy_1c0,rhianna_1c0,[234],[308],shorfy,rhianna,False,2,random_forest_zscore_standard_k50_umap_nc5_sex...,random_forest,...,None,None,False,50,NaN,NaN,NaN,NaN,NaN,NaN
3,shorfy_1c0,rhianna_1c0,[234],[317],shorfy,rhianna,False,0,random_forest_zscore_standard_k50_umap_nc5_sex...,random_forest,...,None,None,False,50,NaN,NaN,NaN,NaN,NaN,NaN
4,shorfy_2c0,rhianna_2c0,"[230, 228]","[310, 311]",shorfy,rhianna,False,2,random_forest_zscore_standard_k50_umap_nc5_sex...,random_forest,...,None,None,False,50,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14309,anton_6c0,paulinchen_6c0,"[357, 353, 358, 355, 354, 356]","[248, 249, 251, 250, 247, 246]",anton,paulinchen,False,2,random_forest_no_out_robust_k2_lda_nc2_sex_off,random_forest,...,None,None,False,2,NaN,NaN,NaN,NaN,NaN,NaN
14310,anton_6c0,paulinchen_6c0,"[363, 364, 362, 365, 360, 361]","[245, 244, 240, 242, 241, 243]",anton,paulinchen,False,1,random_forest_no_out_robust_k2_lda_nc2_sex_off,random_forest,...,None,None,False,2,NaN,NaN,NaN,NaN,NaN,NaN
14311,anton_6c0,paulinchen_6c0,"[363, 364, 362, 365, 360, 361]","[248, 249, 251, 250, 247, 246]",anton,paulinchen,False,2,random_forest_no_out_robust_k2_lda_nc2_sex_off,random_forest,...,None,None,False,2,NaN,NaN,NaN,NaN,NaN,NaN
14312,anton_7c0,paulinchen_7c0,"[359, 357, 356, 353, 354, 358, 355]","[257, 254, 256, 253, 255, 252, 258]",anton,paulinchen,False,0,random_forest_no_out_robust_k2_lda_nc2_sex_off,random_forest,...,None,None,False,2,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
import pandas as pd
import seaborn as sns  # nur für die Heatmap-Matrix

df_pairs = pd.DataFrame(comparisons)
mat = pd.crosstab(df_pairs.ind_a, df_pairs.ind_b)
sns.heatmap(mat, annot=True, fmt="d")
plt.title("Pair-Count Heatmap")
plt.show()

NameError: name 'comparisons' is not defined